> **Change log**
>
> **v1 (Nov 22, Shuo, agreed with Kane):** LIME results were read with `exp.as_list()`, which returns condition strings such as `"PAY_1=2"` instead of feature names, so SHAP–LIME agreement was almost always 0. All LIME results now go through `lime_feature_weights()`, which maps `exp.as_map()` back to feature names. Client `ID`s were added to the result files.
>
> **v2 (Nov 23, Shuo, agreed with Kane):** changes after a review of the experimental design. Changed code is marked `# [v2]`.
> 1. **Larger sample with a validation/test split.** 100 clients (17 defaulters) is too few for Part C to choose and evaluate the flag. The sample is now 200 validation clients plus 300 test clients. The 300 test clients include the original 100, so earlier IDs still apply. Every result file has a `split` column (`val` or `test`).
> 2. **Sensitivity is tested for both models** (it was Logistic Regression only), with SHAP and LIME.
> 3. **Different perturbation for each client.** The perturbation function reset the same seed for every client, so all clients got the same perturbation pattern. The seed now depends on the client ID.
> 4. **LIME randomness separated from input sensitivity.** The original and perturbed inputs are explained with the same LIME random seed, so the measured change comes from the input change, not from LIME's own sampling.
> 5. SHAP values are computed for the sampled clients only (faster); settings are unchanged.
> 6. The demo success case must be a client who actually defaulted and was correctly flagged.

### 1: Environment Setup & Configuration Loading

This cell installs required packages (SHAP, LIME), imports all dependencies, sets up project file paths, and loads the Part A configuration file. This ensures all parameters (feature list, random seed, decision thresholds) are perfectly aligned with the baseline models for reproducibility.

In [1]:
# Install all required dependencies (xgboost added for model loading)
!pip -q install shap lime xgboost

import os
import json
import numpy as np
import pandas as pd
import joblib
import shap
import lime
import lime.lime_tabular

# File paths aligned with actual project directory structure
DATA_DIR = "../01_Data/processed"
MODEL_DIR = "../03_Results/models"
RESULT_DIR = "../03_Results/tables/PartB_tables"   # [v2] results go to the Part B subfolder
FIG_DIR = "../03_Results/figures"        # Plots go here
CONFIG_DIR = "../03_Results"             # config.json is at root of 03_Results

# Create output directories if they do not exist
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Load Part A configuration with corrected path
with open(os.path.join(CONFIG_DIR, "config.json")) as f:
    config = json.load(f)

FEATURES = config["features"]
TARGET = config["target"]
SEED = config["seed"]
THRESHOLDS = config["thresholds_chosen_on_validation"]

print(f"Number of input features: {len(FEATURES)}")
print(f"Feature list: {FEATURES}")

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of input features: 19
Feature list: ['LIMIT_BAL', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


### 2: Data & Model Loading + Alignment Validation

This cell loads the fixed train/validation/test splits and the two pre-trained models from Part A, and checks that predictions match Part A's saved results.

In [2]:
# Load pre-split datasets from Part A
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))     # [v2]
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

X_train = train_df[FEATURES]
X_test  = test_df[FEATURES]

# Load pre-trained baseline models
model_lr  = joblib.load(os.path.join(MODEL_DIR, "logistic_regression.joblib"))
model_xgb = joblib.load(os.path.join(MODEL_DIR, "xgboost.joblib"))
MODELS = {"lr": model_lr, "xgb": model_xgb}

# ========== Critical validation: match Part A predictions ==========
sample_pred_lr  = model_lr.predict_proba(X_test.iloc[:5])[:, 1]
sample_pred_xgb = model_xgb.predict_proba(X_test.iloc[:5])[:, 1]
print("Logistic Regression first 5 predicted probabilities:", sample_pred_lr.round(4))
print("XGBoost first 5 predicted probabilities:          ", sample_pred_xgb.round(4))
print(" If values match predictions_test.csv, loading is successful")

Logistic Regression first 5 predicted probabilities: [0.6131 0.1839 0.1821 0.4945 0.2504]
XGBoost first 5 predicted probabilities:           [0.6206 0.1581 0.1073 0.7088 0.2574]
 If values match predictions_test.csv, loading is successful


### 3: Explained clients (validation + test)

**[v2]** 300 test clients (the original 100 drawn with seed 42, plus 200 more drawn with seed 43 from the remaining test clients) and 200 validation clients (seed 42). Part A's fidelity tests use exactly the same clients. The list is saved to `explain_sample.csv`.

In [3]:
# [v2] Same sample definition is used in Part A step 2
def build_explain_sample(val_df, test_df, seed=SEED):
    np.random.seed(seed)
    test_pos_orig = np.random.choice(len(test_df), 100, replace=False)       # the original 100
    rest = np.setdiff1d(np.arange(len(test_df)), test_pos_orig)
    test_pos_new = np.random.RandomState(seed + 1).choice(rest, 200, replace=False)
    np.random.seed(seed)
    val_pos = np.random.choice(len(val_df), 200, replace=False)
    parts = []
    for split, df, pos in [("val", val_df, val_pos), ("test", test_df, np.concatenate([test_pos_orig, test_pos_new]))]:
        p = df.iloc[pos].copy()
        p.insert(0, "row_index", pos)
        p.insert(0, "split", split)
        parts.append(p)
    return pd.concat(parts, ignore_index=True)

sample_df = build_explain_sample(val_df, test_df)
sample_df[["split", "row_index", "ID", TARGET]].to_csv(os.path.join(RESULT_DIR, "explain_sample.csv"), index=False)
X_sample = sample_df[FEATURES]
print(sample_df.groupby("split")[TARGET].agg(["count", "sum"]).rename(columns={"count": "clients", "sum": "defaulters"}))

       clients  defaulters
split                     
test       300          69
val        200          43


### 4: Utility Functions & Categorical Feature Definition

Reusable metric functions (Jaccard similarity, sign flip rate, top features) and the categorical feature indices used by LIME.

In [4]:
def calculate_jaccard(list1, list2):
    """Compute Jaccard similarity between two lists of top features"""
    set1, set2 = set(list1), set(list2)
    union_size = len(set1 | set2)
    return len(set1 & set2) / union_size if union_size != 0 else 0

def count_sign_flip(dict1, dict2):
    """Calculate proportion of features with flipped sign of weight"""
    common_feats = set(dict1.keys()) & set(dict2.keys())
    flip_count = sum(1 for f in common_feats if np.sign(dict1[f]) != np.sign(dict2[f]))
    flip_ratio = flip_count / len(common_feats) if common_feats else 0
    flip_features = [f for f in common_feats if np.sign(dict1[f]) != np.sign(dict2[f])]
    return flip_ratio, flip_features

def get_top_features(feat_weight_dict, top_n=5):
    """Extract top-N feature names sorted by absolute weight value"""
    sorted_feats = sorted(feat_weight_dict.items(), key=lambda x: abs(x[1]), reverse=True)
    return [feat for feat, weight in sorted_feats[:top_n]]

def lime_feature_weights(exp, label=1):  # [Fix v1] map LIME results to feature names
    """Return {feature name: weight} for a LIME explanation (as_list() returns condition strings instead)"""
    return {FEATURES[j]: w for j, w in exp.as_map()[label]}

# Identify categorical feature indices (PAY_1 to PAY_6 are discrete repayment statuses)
categorical_idx = [
    i for i, feat in enumerate(FEATURES)
    if feat.startswith("PAY_") and not feat.startswith("PAY_AMT")
]
print("Categorical feature indices:", categorical_idx)

Categorical feature indices: [1, 2, 3, 4, 5, 6]


### 5: SHAP Explanation Generation

SHAP explanations for both models with a fixed 150-client background from the training set, in log-odds units. SHAP sub-samples the background to 100 clients.

In [5]:
# Sample 150 rows from training set as SHAP background (fixed random seed)
np.random.seed(SEED)
background_idx = np.random.choice(len(X_train), 150, replace=False)
X_background = X_train.iloc[background_idx]

# Logistic Regression: LinearExplainer on the scaled features (log-odds)
scaler = model_lr.named_steps["scaler"]
lr_classifier = model_lr.named_steps["clf"]
explainer_lr = shap.LinearExplainer(lr_classifier, scaler.transform(X_background))

# XGBoost: TreeExplainer with raw (log-odds) output
explainer_xgb = shap.TreeExplainer(model_xgb, X_background, model_output="raw")

def shap_weights(model_key, X_rows):
    """SHAP values (log-odds) for a DataFrame of rows -> array"""
    if model_key == "lr":
        return explainer_lr.shap_values(scaler.transform(X_rows))
    return explainer_xgb.shap_values(X_rows)

shap_values = {k: shap_weights(k, X_sample) for k in MODELS}   # [v2] sampled clients only

shap_config = {
    "background_size": 150,
    "effective_background_size": 100,  # SHAP sub-samples backgrounds larger than 100 rows
    "background_source": "training_set",
    "output_unit": "log_odds",
    "note_for_xgboost": "TreeExplainer uses model_output='raw' which returns log-odds values",
    "random_seed": SEED
}
with open(os.path.join(RESULT_DIR, "shap_config.json"), "w") as f:
    json.dump(shap_config, f, indent=2)
print(" SHAP explanations generated successfully")

Background dataset has 150 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=150 when initializing the masker.


Background dataset has 150 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=150 when initializing the masker.


 SHAP explanations generated successfully


### 6: LIME Stability Experiment

LIME is run with 5 different random seeds on each sampled client. We report the average pairwise Jaccard similarity of the top-5 features and the sign flip rate, and identify the least stable clients (failure cases).

In [6]:
LIME_SEEDS = [42, 123, 256, 789, 1024]
TOP_N_FEATURES = 5
LIME_NUM_SAMPLES = 2000  # Reduced from default 5000 for speed

def build_lime_explainers(seeds):
    """Pre-construct LIME explainers for all seeds"""
    return {seed: lime.lime_tabular.LimeTabularExplainer(
                training_data=np.array(X_train), feature_names=FEATURES,
                categorical_features=categorical_idx, class_names=["No Default", "Default"],
                mode="classification", random_state=seed)
            for seed in seeds}

lime_explainers = build_lime_explainers(LIME_SEEDS)

def run_lime_stability_fast(explainers, model, X_sample_row):
    all_top_feats, all_weight_dicts = [], []
    for seed in LIME_SEEDS:
        exp = explainers[seed].explain_instance(
            data_row=X_sample_row.values, predict_fn=model.predict_proba,
            num_features=len(FEATURES), num_samples=LIME_NUM_SAMPLES)
        weight_dict = lime_feature_weights(exp)
        all_weight_dicts.append(weight_dict)
        all_top_feats.append(get_top_features(weight_dict, TOP_N_FEATURES))
    jaccard_scores, sign_flip_ratios = [], []
    for i in range(len(all_top_feats)):
        for j in range(i + 1, len(all_top_feats)):
            jaccard_scores.append(calculate_jaccard(all_top_feats[i], all_top_feats[j]))
            sign_flip_ratios.append(count_sign_flip(all_weight_dicts[i], all_weight_dicts[j])[0])
    return {"avg_jaccard": np.mean(jaccard_scores), "avg_sign_flip": np.mean(sign_flip_ratios),
            "all_top_feats": all_top_feats}

stability = {k: [] for k in MODELS}
for i, row in sample_df.iterrows():   # [v2] all 500 sampled clients
    for k, m in MODELS.items():
        res = run_lime_stability_fast(lime_explainers, m, row[FEATURES])
        stability[k].append({"split": row["split"], "row_index": row["row_index"], "ID": int(row["ID"]), **res})

stab_df = {k: pd.DataFrame(v) for k, v in stability.items()}
for k, d in stab_df.items():
    d.to_csv(os.path.join(RESULT_DIR, f"lime_stability_{k}.csv"), index=False)
    print(f"{k}: mean Jaccard {d['avg_jaccard'].mean():.3f}, sign flip {d['avg_sign_flip'].mean():.3f}")

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

... (299 outputs left out)

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

lr: mean Jaccard 0.620, sign flip 0.264
xgb: mean Jaccard 0.569, sign flip 0.167


/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


### 7: Input Sensitivity Experiment

Small, realistic perturbations (±5% and ±15%) are applied to the numeric features; repayment-status codes stay unchanged. We measure how much the SHAP and LIME explanations change (top-5 Jaccard and sign flip rate) for **both models**.

**[v2]** Each client gets its own perturbation (seed depends on the client ID). LIME is run with the **same random seed** on the original and the perturbed input, so the measured change comes from the input, not from LIME's sampling.

In [7]:
NUMERIC_COLS = [f for i, f in enumerate(FEATURES) if i not in categorical_idx]

def perturb_numeric_features(X_row, perturb_ratio, client_id):
    """[v2] Proportional perturbation of numeric features; a different pattern for each client"""
    rng = np.random.default_rng(SEED + int(client_id))
    X_perturbed = X_row.astype(float).copy()
    for col in NUMERIC_COLS:
        delta = X_row[col] * perturb_ratio * rng.uniform(-1, 1)
        X_perturbed[col] = max(0, X_row[col] + delta)   # amounts stay non-negative
    return X_perturbed

def measure_explanation_change(original_weights, perturbed_weights, top_n=5):
    orig_top = get_top_features(original_weights, top_n)
    pert_top = get_top_features(perturbed_weights, top_n)
    return calculate_jaccard(orig_top, pert_top), count_sign_flip(original_weights, perturbed_weights)[0]

lime_base = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train), feature_names=FEATURES, categorical_features=categorical_idx,
    class_names=["No Default", "Default"], mode="classification", random_state=SEED)

def lime_fixed_seed(model, row_values):
    """[v2] LIME with the random state reset, so repeated calls sample identically"""
    lime_base.random_state = np.random.RandomState(SEED)
    exp = lime_base.explain_instance(row_values, model.predict_proba, num_features=len(FEATURES))
    return lime_feature_weights(exp)

PERTURB_RATIOS = [0.05, 0.15]
sensitivity_results = []
lime_original = {k: {} for k in MODELS}   # reused for the agreement analysis below

for i, row in sample_df.iterrows():
    x = row[FEATURES]
    cid = int(row["ID"])
    for k, m in MODELS.items():
        orig_shap = dict(zip(FEATURES, shap_values[k][i]))
        orig_lime = lime_fixed_seed(m, x.values.astype(float))
        lime_original[k][cid] = orig_lime
        for ratio in PERTURB_RATIOS:
            xp = perturb_numeric_features(x, ratio, cid)
            pert_shap = dict(zip(FEATURES, shap_weights(k, pd.DataFrame([xp], columns=FEATURES))[0]))
            pert_lime = lime_fixed_seed(m, xp.values.astype(float))
            sj, sf = measure_explanation_change(orig_shap, pert_shap)
            lj, lf = measure_explanation_change(orig_lime, pert_lime)
            sensitivity_results.append({"split": row["split"], "row_index": row["row_index"], "ID": cid,
                                        "model": k, "perturb_ratio": ratio,
                                        "shap_jaccard": sj, "shap_sign_flip": sf,
                                        "lime_jaccard": lj, "lime_sign_flip": lf})

sens_df = pd.DataFrame(sensitivity_results)
sens_df.to_csv(os.path.join(RESULT_DIR, "sensitivity_results.csv"), index=False)
print(sens_df.groupby(["model", "perturb_ratio"])[["shap_jaccard", "shap_sign_flip", "lime_jaccard", "lime_sign_flip"]].mean().round(3))

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


... (341 outputs left out)

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/ut

/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


                     shap_jaccard  shap_sign_flip  lime_jaccard  \
model perturb_ratio                                               
lr    0.05                  0.983           0.004         0.878   
      0.15                  0.952           0.011         0.873   
xgb   0.05                  0.916           0.017         0.884   
      0.15                  0.860           0.031         0.862   

                     lime_sign_flip  
model perturb_ratio                  
lr    0.05                    0.056  
      0.15                    0.061  
xgb   0.05                    0.035  
      0.15                    0.038  


### 8: SHAP-LIME Agreement Analysis

For each client and model, the overlap of the top-5 features from SHAP and LIME (LIME with seed 42, 5,000 samples, the same explanation as the unperturbed input in Section 7).

In [8]:
agreement_results = []
for i, row in sample_df.iterrows():
    cid = int(row["ID"])
    rec = {"split": row["split"], "row_index": row["row_index"], "ID": cid}
    for k in MODELS:
        shap_w = dict(zip(FEATURES, shap_values[k][i]))
        lime_w = lime_original[k][cid]
        rec[f"agreement_jaccard_{k}"] = calculate_jaccard(get_top_features(shap_w), get_top_features(lime_w))
        flip, _ = count_sign_flip(shap_w, lime_w)
        rec[f"agreement_sign_flip_{k}"] = flip
    agreement_results.append(rec)

agree_df = pd.DataFrame(agreement_results)
agree_df.to_csv(os.path.join(RESULT_DIR, "shap_lime_agreement.csv"), index=False)
print(agree_df.groupby("split")[["agreement_jaccard_lr", "agreement_jaccard_xgb"]].mean().round(3))

       agreement_jaccard_lr  agreement_jaccard_xgb
split                                             
test                  0.625                  0.434
val                   0.622                  0.427


### 9: Correlated Feature Impact Analysis

How highly correlated feature groups (PAY repayment status series, BILL_AMT bill amount series) relate to how SHAP spreads weight across them.

In [9]:
corr_matrix = X_train.corr()
pay_cols = [f for f in FEATURES if f.startswith("PAY_") and not f.startswith("PAY_AMT")]
bill_cols = [f for f in FEATURES if f.startswith("BILL_AMT")]

pay_avg_corr = corr_matrix.loc[pay_cols, pay_cols].mean().mean()
bill_avg_corr = corr_matrix.loc[bill_cols, bill_cols].mean().mean()
pay_shap_std_lr = np.std(np.abs(shap_values["lr"][:, [FEATURES.index(c) for c in pay_cols]]), axis=1).mean()
bill_shap_std_lr = np.std(np.abs(shap_values["lr"][:, [FEATURES.index(c) for c in bill_cols]]), axis=1).mean()

corr_analysis = {
    "pay_series_avg_correlation": round(float(pay_avg_corr), 4),
    "bill_amt_series_avg_correlation": round(float(bill_avg_corr), 4),
    "pay_series_shap_weight_std_lr": round(float(pay_shap_std_lr), 4),
    "bill_series_shap_weight_std_lr": round(float(bill_shap_std_lr), 4),
    "key_conclusion": "Highly correlated sequential features share credit in the explanations, which can reduce stability and cross-method agreement"
}
with open(os.path.join(RESULT_DIR, "correlated_features_analysis.json"), "w") as f:
    json.dump(corr_analysis, f, indent=2)
print(corr_analysis)

{'pay_series_avg_correlation': 0.7139, 'bill_amt_series_avg_correlation': 0.909, 'pay_series_shap_weight_std_lr': 0.1664, 'bill_series_shap_weight_std_lr': 0.0601, 'key_conclusion': 'Highly correlated sequential features share credit in the explanations, which can reduce stability and cross-method agreement'}


### 10: Demo cases and final export

**[v2]** Demo cases are chosen from the **test** clients:
- **Failure case:** the least stable LIME explanation for each model.
- **Success case:** among test clients who actually defaulted and were correctly flagged by XGBoost, the one with the highest LIME stability and SHAP–LIME agreement (ties broken by predicted risk).

A SHAP plot is saved for each demo client.

In [10]:
import matplotlib.pyplot as plt

test_mask = stab_df["lr"]["split"] == "test"
worst_lr = stab_df["lr"][test_mask].sort_values("avg_jaccard").iloc[0]
worst_xgb = stab_df["xgb"][stab_df["xgb"]["split"] == "test"].sort_values("avg_jaccard").iloc[0]

cand = sample_df[sample_df["split"] == "test"][["ID", TARGET] + FEATURES].copy()
cand["proba_xgb"] = model_xgb.predict_proba(cand[FEATURES])[:, 1]
cand = cand.merge(stab_df["xgb"][["ID", "avg_jaccard"]], on="ID").merge(agree_df[["ID", "agreement_jaccard_xgb"]], on="ID")
cand = cand[(cand["proba_xgb"] >= THRESHOLDS["XGBoost"]) & (cand[TARGET] == 1)]   # correctly flagged defaulters
success = cand.sort_values(["avg_jaccard", "agreement_jaccard_xgb", "proba_xgb"], ascending=False).iloc[0]

demo_cases = {
    "success_case_xgb": {"ID": int(success["ID"]), "proba": round(float(success["proba_xgb"]), 4),
                         "lime_stability_jaccard": round(float(success["avg_jaccard"]), 3),
                         "shap_lime_agreement": round(float(success["agreement_jaccard_xgb"]), 3)},
    "failure_case_lr": {"ID": int(worst_lr["ID"]), "lime_stability_jaccard": round(float(worst_lr["avg_jaccard"]), 3)},
    "failure_case_xgb": {"ID": int(worst_xgb["ID"]), "lime_stability_jaccard": round(float(worst_xgb["avg_jaccard"]), 3)},
}
with open(os.path.join(RESULT_DIR, "demo_cases.json"), "w") as f:
    json.dump(demo_cases, f, indent=2)
print(json.dumps(demo_cases, indent=2))

# SHAP plots (XGBoost, log-odds) for the success case and the XGBoost failure case
for name, cid in [("success", demo_cases["success_case_xgb"]["ID"]), ("failure", demo_cases["failure_case_xgb"]["ID"])]:
    i = sample_df.index[sample_df["ID"] == cid][0]
    shap.force_plot(explainer_xgb.expected_value, shap_values["xgb"][i], sample_df.loc[i, FEATURES],
                    show=False, matplotlib=True)
    plt.savefig(os.path.join(FIG_DIR, f"shap_demo_{name}_xgb.png"), dpi=200, bbox_inches="tight")
    plt.close()
print(" All result files and demo assets saved successfully")

{
  "success_case_xgb": {
    "ID": 17775,
    "proba": 0.7515,
    "lime_stability_jaccard": 1.0,
    "shap_lime_agreement": 1.0
  },
  "failure_case_lr": {
    "ID": 25906,
    "lime_stability_jaccard": 0.262
  },
  "failure_case_xgb": {
    "ID": 6391,
    "lime_stability_jaccard": 0.272
  }
}


 All result files and demo assets saved successfully
